<a href="https://colab.research.google.com/github/BenFreilichUVA/ds2002-fa26/blob/main/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [96]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [97]:
# TODO
df['revenue'] = df['qty'] * df['price']
df.head()

TotalRevenue = df['revenue'].sum()
TotalUnitsSold = df['qty'].sum()

print(f'The total revenue was ${TotalRevenue} and the total units sold was {TotalUnitsSold}.')

The total revenue was $8520.0 and the total units sold was 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [98]:
# TODO
by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).to_frame()
by_category['share_pct'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(2)
display(by_category)

print("Food held the largest share of revenue with more than half of the share, followed by merch, drink, and finally rain gear.")

,revenue,share_pct
category,,
Food,4293.0,50.39
Merch,1771.5,20.79
Drink,1554.0,18.24
RainGear,901.5,10.58


Food held the largest share of revenue with more than half of the share, followed by merch, drink, and finally rain gear.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [99]:
# TODO
by_vendor = df.groupby('vendor_id')['revenue'].agg(orders='count', total_revenue='sum', avg_order_revenue='mean').sort_values('avg_order_revenue', ascending=False)
display(by_vendor)
print("Vendor 01 had the highest average order revenue. While they are all relatively close, Vendor 01 is higher than the second highest by a little less than a dollar, \nhowever all vendors are within an order range of 93-108, so the ranking may not end up being meaningful.")

,orders,total_revenue,avg_order_revenue
vendor_id,,,
V-01,94,2124.0,22.595745
V-18,108,2349.0,21.750000
V-05,93,1914.0,20.580645
V-10,105,2133.0,20.314286


Vendor 01 had the highest average order revenue. While they are all relatively close, Vendor 01 is higher than the second highest by a little less than a dollar, 
however all vendors are within an order range of 93-108, so the ranking may not end up being meaningful.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [100]:
# TODO
MerchShare = by_category.loc['Merch', 'revenue'] / df['revenue'].sum()
print(f"Merch accounts for {MerchShare:.1%} of the revenue.")

Merch accounts for 20.8% of the revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [101]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

print('Row count unchanged:', len(joined) == len(df))
print('Revenue total unchanged:', joined['revenue'].sum() == df['revenue'].sum())

unmatched = joined[joined['vendor_name'].isna()]
print('Unmatched vendor_id:', unmatched['vendor_id'].unique())
print('The number of unmatched orders is:', len(unmatched), '| The amount of unmatched revenue is:', unmatched['revenue'].sum(), '| The share of the total revenue is:', f"{unmatched['revenue'].sum() / df['revenue'].sum():.1%}")

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (V-18)')

Row count unchanged: True
Revenue total unchanged: True
Unmatched vendor_id: ['V-18']
The number of unmatched orders is: 108 | The amount of unmatched revenue is: 2349.0 | The share of the total revenue is: 27.6%


**The unmatched vendor, and what I did about it:** I simply marked the vendor as unknown, while keeping all of the orders. Given the vendor makes up more than a quarter of the revenue, it would not be feasible to drop the vendor and its orders, as they are more than material to the total financial performance. Because there is only one unknown vendor, we know all of the unknown orders are coming from the same vendor, despite not having an officially identifying name for them. Keeping the vendor also ensures the pivot table does not lose accuracy. Given the materiality, losing 108 orders and $2349 of revenue would be substantial for trying to make decisions based off of financial variables.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [102]:
# TODO
display(pd.pivot_table(joined, values='revenue', index='vendor_name', columns='category', aggfunc='sum', margins=True))
print("As can be seen in the pivot table, the unknown vendor is still accounting for a large share of revenue, notably, they are the largest vendor by revenue share. \nKeeping them in the table ensures the revenue report remains accurate.")

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown (V-18),582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


As can be seen in the pivot table, the unknown vendor is still accounting for a large share of revenue, notably, they are the largest vendor by revenue share. 
Keeping them in the table ensures the revenue report remains accurate.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [103]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) Given food makes up the majority of the revenue, I would encourage vendors to prioritize food availability and marketing towards food. Hoos Burgers sold a very low amount of drinks, only 171, relative to the other vendors compared to their food, I would possibly encourage them to create a combo deal to increase drink sales. Cav Merch North did not end up selling very much merch, just 400.5, possibly indicating a poor marketing campaign or bad merch products, something that could be improved. Finally, I would encourage the stadium executives as a whole to learn who the unknown vendor is, as that would provide more context into which to make decisions with.

b) The least trustworthy answer is Q3. There isn't an exceptionally strong difference in the average revenue per order between the different vendors. The range from roughly 22.60 to 20.30 is not huge, and could simply be due to random selection. If the sample size was much larger, or the range was wider, I would feel more confident in making a definitive statement based off of these answers.